In [ ]:
from PIL import Image
import os
import numpy as np
import random
from glob import glob
from tqdm import tqdm
import shutil
from scipy.io import loadmat, savemat 
import csv
import matplotlib.pyplot as plt
import json
from copy import deepcopy
def create_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)
        
base_json={
    "file_name": "",
    "cordinates": [], #[class, x_min, y_min, x_max, y_max]
}

Lizard processing

In [ ]:
file_path='../../data/HnE_cell_detect/Lizard/'
image_list=glob(file_path+'lizard_images*/Lizard_Images*/*.png')
label_list=glob(file_path+'lizard_labels/Lizard_Labels/Labels/*.mat')
save_file_path='../../data/HnE_cell_detect/total_data/'
create_dir(save_file_path+'images/')
create_dir(save_file_path+'labels/')
class_label={1:"Neutrophil",
             2:"Epithelial",
             3:"Lymphocyte",
             4:"Plasma",
             5:"Eosinophil",
             6:"Connective tissue",}
class_count={
    "Neutrophil":0,
    "Epithelial":0,
    "Lymphocyte":0,
    "Plasma":0,
    "Eosinophil":0,
    "Connective tissue":0,
}
for idx in tqdm(range(len(image_list))):
    json_data=deepcopy(base_json)
    json_data['file_name'] = os.path.basename(label_list[idx]).replace('.mat', '.png')
    data = loadmat(label_list[idx])
    
    for i in range(data['bbox'].shape[0]):
        temp_cordinates=[0,0,0,0,0]
        temp_cordinates[0] = int(data['class'][i][0])
        temp_cordinates[1] = int(data['bbox'][i][0])
        temp_cordinates[2] = int(data['bbox'][i][2])
        temp_cordinates[3] = int(data['bbox'][i][1]-data['bbox'][i][0])
        temp_cordinates[4] = int(data['bbox'][i][3]-data['bbox'][i][2])
        json_data['cordinates'].append(temp_cordinates)
        class_count[class_label[temp_cordinates[0]]]+=1
    shutil.copy(image_list[idx], save_file_path+'images/'+os.path.basename(image_list[idx]))
    with open(save_file_path+'labels/'+os.path.basename(label_list[idx]).replace('.mat', '.json'), 'w') as f:
        json.dump(json_data, f, indent=2)


In [ ]:
# total_data 폴더의 이미지와 라벨을 overlap 폴더에 저장
import matplotlib.patches as patches

data_path = '../../data/HnE_cell_detect/total_data/'
image_files = sorted(glob(data_path + 'images/*.png'))
label_files = sorted(glob(data_path + 'labels/*.json'))

# overlap 폴더 생성
overlap_dir = data_path + 'overlap/'
create_dir(overlap_dir)

print(f"총 이미지 수: {len(image_files)}")
print(f"총 라벨 수: {len(label_files)}")
print(f"저장 경로: {overlap_dir}")

# 클래스별 색상 정의 (HnE 데이터 6클래스)
class_colors = {
    1: ('orange', 'Neutrophil'),
    2: ('green', 'Epithelial'),
    3: ('red', 'Lymphocyte'),
    4: ('skyblue', 'Plasma'),
    5: ('blue', 'Eosinophil'),
    6: ('yellow', 'Connective tissue')
}

# 전체 데이터셋 통계
print("\n" + "="*60)
print("📊 전체 데이터셋 통계")
print("="*60)

total_cells = 0
all_class_counts = {i: 0 for i in range(1, 7)}

for label_file in tqdm(label_files, desc="라벨 분석 중"):
    with open(label_file, 'r') as f:
        label_data = json.load(f)
    
    for coord in label_data['cordinates']:
        class_id = coord[0]
        all_class_counts[class_id] += 1
        total_cells += 1

print(f"\n총 세포 수: {total_cells:,}")
print(f"\n클래스별 분포:")
for class_id in range(1, 7):
    count = all_class_counts[class_id]
    percentage = (count / total_cells * 100) if total_cells > 0 else 0
    print(f"  {class_colors[class_id][1]:20s}: {count:6,} cells ({percentage:5.2f}%)")

print(f"\n평균 세포 수/이미지: {total_cells/len(label_files):.1f}")
print("="*60)

# 모든 이미지에 대해 overlap 이미지 생성 및 저장
print(f"\n🖼️ Overlap 이미지 생성 중...")

for idx in tqdm(range(len(image_files)), desc="Overlap 저장"):
    # 이미지 로드
    img = Image.open(image_files[idx])
    img_array = np.array(img)
    
    # 라벨 로드
    with open(label_files[idx], 'r') as f:
        label_data = json.load(f)
    
    # 그림 생성
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img_array)
    ax.set_title(f'{os.path.basename(image_files[idx])} ({len(label_data["cordinates"])} cells)', 
                 fontsize=12)
    ax.axis('off')
    
    # 각 cell의 중심점을 point로 표시
    for coord in label_data['cordinates']:
        class_id = coord[0]
        y_min = coord[1]
        x_min = coord[2]
        h = coord[3]
        w = coord[4]
        
        # 중심점 계산
        center_x = x_min + w / 2
        center_y = y_min + h / 2
        
        # 클래스 색상
        color, _ = class_colors.get(class_id, ('white', 'Unknown'))
        
        # 중심점 표시
        ax.scatter( center_x,center_y, c=color, s=20, alpha=0.8, edgecolors='black', linewidth=0.5)
    
    # 파일로 저장
    save_path = overlap_dir + os.path.basename(image_files[idx])
    plt.savefig(save_path, dpi=150, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)

print(f"\n✅ 완료! {len(image_files)}개의 overlap 이미지가 {overlap_dir}에 저장되었습니다.")